# 🎯 Navigator-OptiLite — YOLO Live Detection
**AMD MI300X · ROCm 7.2 · YOLOv8 · 3fps Webcam Stream**

---

> ⚠️ Allow webcam access in your browser.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 1 - Imports & GPU Verification                ║
# ╚══════════════════════════════════════════════════════╝

import torch
import cv2
import numpy as np
import base64
from PIL import Image
from io import BytesIO
from IPython.display import display, HTML
from ultralytics import YOLO
import ipywidgets as widgets

# ── We Check the GPU ──────────────────────────────────────────────
if torch.cuda.is_available():
    device = 'cuda'
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ GPU : {gpu_name}')
    print(f'✓ VRAM: {vram:.0f} GB')
    print(f'✓ ROCm: {torch.version.hip}')
else:
    device = 'cpu'
    print('⚠ No GPU found - running on CPU')
    print('  If you expected GPU, check: rocm-smi in terminal')

print(f'\nDevice set to: {device}')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 2 - Semantic Color Map                        ║
# ╚══════════════════════════════════════════════════════╝
# Colors stored as BGR (OpenCV format)
# Tailwind-inspired shades

# ── BGR color constants (We use BGR not RGB because of OpenCV) ────────────────────────────────────
BLUE_500    = (219, 130,  66)   # humans
RED_900     = ( 30,  27, 127)   # dangerous animals
ORANGE_400  = ( 49, 130, 247)   # safe animals
GREEN_300   = ( 134, 239, 172)  # car door   (lightest)
GREEN_500   = ( 74,  222, 128)  # door generic
GREEN_600   = ( 22,  163,  74)  # house door (mid)
GREEN_700   = ( 21,  128,  61)  # bus door
GREEN_900   = ( 20,   83,  45)  # darkest door
GRAY        = (200,  200, 200)  # fallback

# ── Class sets ─────────────────────────────────────────────
DANGEROUS_ANIMALS = {'bear', 'elephant'}
SAFE_ANIMALS      = {
    'dog', 'cat', 'horse', 'cow', 'sheep',
    'bird', 'giraffe', 'zebra'
}

# Direct overrides (COCO class name → BGR)
CLASS_COLOR_MAP = {
    'person':     BLUE_500,
    'bear':       RED_900,
    'elephant':   RED_900,
    'snake':      RED_900,
    'dog':        ORANGE_900,
    'cat':        ORANGE_800,
    'horse':      ORANGE_700,
    'cow':        ORANGE_600,
    'sheep':      ORANGE_500,
    'bird':       PINK_500,
    'giraffe':    ORANGE_400,
    'zebra':      ORANGE_400,
    # door subtype detection later with LLaVA
    # For now all doors detected via context get GREEN_600
}

def get_color(class_name: str) -> tuple:
    """Return BGR color tuple for a YOLO class name."""
    c = class_name.lower().strip()
    if c in CLASS_COLOR_MAP:
        return CLASS_COLOR_MAP[c]
    return GRAY

# ── Quick sanity check ─────────────────────────────────────
tests = ['person', 'bear', 'dog', 'elephant', 'car', 'chair']
for t in tests:
    print(f'  {t:12s} → {get_color(t)}')
print('\nColor map ready ✓')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 3 - Load YOLOv8 Model                         ║
# ╚══════════════════════════════════════════════════════╝

print('Loading YOLOv8x...')
model = YOLO('yolov8x.pt')   # extra large = most accurate, fast on M1300X
model.to(device)

print(f'\nModel: YOLOv8x')
print(f'Device: {device}')
print(f'Classes: {len(model.names)} COCO classes')
print('\nModel ready ✓')

# Warm up the model (first inference is always slow)
print('Warming up...')
dummy = torch.zeros(1, 3, 640, 480).to(device)
_ = model(np.zeros((480, 640, 3), dtype=np.uint8), verbose=False)
print('Warm up done ✓')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 4 - Annotation Engine                         ║
# ╚══════════════════════════════════════════════════════╝

def draw_detections(frame: np.ndarray, results) -> tuple:
    """
    Draw styled detections on frame.
    Returns: (annotated_frame, detection_count, detection_list)
    
    Style:
    - Corner brackets (not solid rect) - clean, HUD-like
    - Colored per semantic class
    - Label pill above box (slides below if too close to top)
    - Confidence filter: skip < 40%
    """
    annotated = frame.copy()
    detections = []

    for result in results:
        if result.boxes is None:
            continue

        for box in result.boxes:
            conf = float(box.conf[0])
            if conf < 0.40:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            cls_id     = int(box.cls[0])
            class_name = model.names[cls_id]
            color      = get_color(class_name)
            
            detections.append({'class': class_name, 'conf': conf})

            # ── Corner brackets ───────────────────────────────
            w, h  = x2 - x1, y2 - y1
            blen  = max(12, min(30, w // 5, h // 5))  # bracket length
            thick = 3

            # TL
            cv2.line(annotated, (x1, y1),        (x1 + blen, y1),        color, thick)
            cv2.line(annotated, (x1, y1),        (x1, y1 + blen),        color, thick)
            # TR
            cv2.line(annotated, (x2, y1),        (x2 - blen, y1),        color, thick)
            cv2.line(annotated, (x2, y1),        (x2, y1 + blen),        color, thick)
            # BL
            cv2.line(annotated, (x1, y2),        (x1 + blen, y2),        color, thick)
            cv2.line(annotated, (x1, y2),        (x1, y2 - blen),        color, thick)
            # BR
            cv2.line(annotated, (x2, y2),        (x2 - blen, y2),        color, thick)
            cv2.line(annotated, (x2, y2),        (x2, y2 - blen),        color, thick)

            # ── Label pill ────────────────────────────────────
            label     = f'{class_name}  {conf:.0%}'
            font      = cv2.FONT_HERSHEY_SIMPLEX
            fscale    = 0.52
            fthick    = 1
            pad       = 5

            (tw, th), baseline = cv2.getTextSize(label, font, fscale, fthick)

            # Position above box; flip below if near top edge
            if y1 - th - pad * 2 - baseline > 0:
                ly1 = y1 - th - pad * 2 - baseline
                ly2 = y1
                ty  = ly2 - baseline - 2
            else:
                ly1 = y2
                ly2 = y2 + th + pad * 2 + baseline
                ty  = ly2 - baseline - 2

            lx1 = x1
            lx2 = x1 + tw + pad * 2

            # Pill background
            cv2.rectangle(annotated, (lx1, ly1), (lx2, ly2), color, -1)
            # White text
            cv2.putText(
                annotated, label,
                (lx1 + pad, ty),
                font, fscale, (255, 255, 255), fthick,
                cv2.LINE_AA
            )

    return annotated, len(detections), detections


def frame_to_b64(frame: np.ndarray, quality: int = 82) -> str:
    """Encode OpenCV BGR frame to base64 JPEG string."""
    ok, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, quality])
    if not ok:
        raise ValueError('Frame encoding failed')
    return base64.b64encode(buf).decode('utf-8')


print('Annotation engine ready ✓')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 5 - Frame Processor (Python side)             ║
# ╚══════════════════════════════════════════════════════╝
# Receives base64 frames from the JS webcam widget,
# runs YOLO, returns annotated frame back to browser.

# Hidden widget - JS writes base64 frame here
frame_input = widgets.Text(
    value='',
    layout=widgets.Layout(display='none')
)
display(frame_input)

frame_counter = [0]  # mutable int for closure

def on_frame(change):
    b64 = change['new']
    if not b64 or len(b64) < 200:
        return

    try:
        # ── Decode ──────────────────────────────────────
        img_bytes = base64.b64decode(b64)
        arr       = np.frombuffer(img_bytes, dtype=np.uint8)
        frame     = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        if frame is None:
            return

        # ── Inference ───────────────────────────────────
        results = model(frame, verbose=False, device=device)

        # ── Annotate ────────────────────────────────────
        annotated, det_count, dets = draw_detections(frame, results)

        # ── Encode result ───────────────────────────────
        result_b64 = frame_to_b64(annotated)

        # Build detection label string for HUD
        if dets:
            det_str = ', '.join(
                f"{d['class']} ({d['conf']:.0%})" for d in dets[:4]
            )
            if len(dets) > 4:
                det_str += f' +{len(dets)-4} more'
        else:
            det_str = 'none'

        frame_counter[0] += 1

        # ── Push annotated frame to browser ─────────────
        escaped_det = det_str.replace("'", "\\'")
        display(HTML(
            f'<script>'
            f'if(window.navUpdate){{'
            f'  window.navUpdate("{result_b64}", {det_count}, \'{escaped_det}\', {frame_counter[0]});'
            f'}}'
            f'</script>'
        ))

    except Exception as e:
        display(HTML(f'<script>if(window.navError)window.navError("{str(e)[:80]}");</script>'))

frame_input.observe(on_frame, names='value')
print('Frame processor hooked ✓')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 6 - Live Detection UI  ★ MAIN CELL ★          ║
# ╚══════════════════════════════════════════════════════╝
# Injects the webcam capture + display UI into the notebook.
# Click ▶ Start Stream to begin.

display(HTML(r'''
<style>
/* ═══ Navigator UI ═══════════════════════════════════════ */
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600&display=swap');

#nav-root {
  font-family: 'JetBrains Mono', 'Courier New', monospace;
  background: #0a0a0a;
  border: 1px solid #1a1a1a;
  border-radius: 14px;
  padding: 18px;
  display: inline-block;
  min-width: 660px;
  user-select: none;
}

/* Header bar */
#nav-header {
  display: flex;
  justify-content: space-between;
  align-items: center;
  margin-bottom: 12px;
}
#nav-title {
  color: #4ade80;
  font-size: 11px;
  letter-spacing: 3px;
  text-transform: uppercase;
  font-weight: 600;
}
#nav-badge {
  background: #111;
  border: 1px solid #222;
  color: #555;
  font-size: 10px;
  padding: 3px 8px;
  border-radius: 20px;
  letter-spacing: 1px;
}
#nav-badge.live {
  border-color: #16a34a;
  color: #4ade80;
  animation: pulse 2s infinite;
}
@keyframes pulse {
  0%, 100% { opacity: 1; }
  50% { opacity: 0.5; }
}

/* Video frame */
#nav-frame {
  position: relative;
  border-radius: 8px;
  overflow: hidden;
  background: #111;
  line-height: 0;
}
#nav-img {
  width: 640px;
  height: 480px;
  display: block;
  object-fit: cover;
}
#nav-overlay {
  position: absolute;
  top: 10px; left: 10px;
  background: rgba(0,0,0,0.6);
  color: #4ade80;
  font-size: 10px;
  padding: 4px 8px;
  border-radius: 4px;
  letter-spacing: 1px;
}

/* Status bar */
#nav-statusbar {
  display: flex;
  justify-content: space-between;
  margin-top: 10px;
  padding: 6px 10px;
  background: #0f0f0f;
  border: 1px solid #1a1a1a;
  border-radius: 6px;
  font-size: 10px;
  color: #444;
}
#nav-det-text { color: #60a5fa; max-width: 400px; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }
#nav-frame-num { color: #333; }

/* Controls */
#nav-controls {
  display: flex;
  gap: 8px;
  margin-bottom: 12px;
}
.nav-btn {
  background: #111;
  border: 1px solid #2a2a2a;
  color: #aaa;
  padding: 7px 16px;
  border-radius: 7px;
  cursor: pointer;
  font-size: 11px;
  font-family: inherit;
  letter-spacing: 1px;
  transition: all 0.15s;
}
.nav-btn:hover { border-color: #4ade80; color: #4ade80; }
.nav-btn.danger:hover { border-color: #ef4444; color: #ef4444; }
.nav-btn:disabled { opacity: 0.3; cursor: not-allowed; }

/* Detection animation: bracket draw-in effect via CSS */
@keyframes bracketIn {
  from { opacity: 0; transform: scale(0.95); }
  to   { opacity: 1; transform: scale(1); }
}
</style>

<div id="nav-root">
  <div id="nav-header">
    <span id="nav-title">⬡ Navigator-OptiLite // YOLO</span>
    <span id="nav-badge">● IDLE</span>
  </div>

  <div id="nav-controls">
    <button class="nav-btn" id="nav-start-btn" onclick="navStartStream()">▶ START STREAM</button>
    <button class="nav-btn danger" id="nav-stop-btn" onclick="navStopStream()" disabled>⏹ STOP</button>
  </div>

  <div id="nav-frame">
    <img id="nav-img"
      src="data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='640' height='480'%3E%3Crect width='640' height='480' fill='%23111'/%3E%3Ctext x='50%25' y='50%25' fill='%23333' font-family='monospace' font-size='13' text-anchor='middle' dominant-baseline='middle'%3EAwaiting stream...%3C/text%3E%3C/svg%3E"
      alt="Detection preview"
    >
    <div id="nav-overlay">YOLOv8n · MI300X · 3fps</div>
  </div>

  <div id="nav-statusbar">
    <span id="nav-det-text">detections: --</span>
    <span id="nav-frame-num">frame: 0</span>
  </div>
</div>

<!-- Hidden video/canvas for capture -->
<video id="nav-video" style="display:none" autoplay playsinline muted></video>
<canvas id="nav-canvas" style="display:none"></canvas>

<script>
(function() {
  const W = 640, H = 480;
  const FPS = 3;
  let active = false;
  let mediaStream = null;

  // ── Find the hidden ipywidgets text input ──────────────
  function getWidgetInput() {
    // ipywidgets text input doesn't have a stable selector,
    // so we look for the most recently added hidden text input
    const inputs = document.querySelectorAll('input[type="text"]');
    for (const inp of inputs) {
      // Find the one in a widget container with display:none
      const container = inp.closest('.widget-text, .widget-hbox, .lm-Widget');
      if (container && (container.style.display === 'none' || getComputedStyle(container).display === 'none')) {
        return inp;
      }
    }
    // Fallback: last text input
    return inputs[inputs.length - 1] || null;
  }

  function sendFrame(b64) {
    const inp = getWidgetInput();
    if (!inp) { console.warn('Navigator: widget input not found'); return; }
    // React-style setter trick to trigger Jupyter's change listener
    const setter = Object.getOwnPropertyDescriptor(HTMLInputElement.prototype, 'value').set;
    setter.call(inp, b64);
    inp.dispatchEvent(new Event('input', { bubbles: true }));
  }

  function captureLoop() {
    if (!active) return;
    const video  = document.getElementById('nav-video');
    const canvas = document.getElementById('nav-canvas');
    canvas.width  = W;
    canvas.height = H;
    const ctx = canvas.getContext('2d');
    // Mirror (selfie view)
    ctx.translate(W, 0);
    ctx.scale(-1, 1);
    ctx.drawImage(video, 0, 0, W, H);
    ctx.setTransform(1,0,0,1,0,0); // reset transform
    const b64 = canvas.toDataURL('image/jpeg', 0.82).split(',')[1];
    sendFrame(b64);
    setTimeout(captureLoop, 1000 / FPS);
  }

  window.navStartStream = async function() {
    if (active) return;
    try {
      mediaStream = await navigator.mediaDevices.getUserMedia({
        video: { width: { ideal: 1280 }, height: { ideal: 720 } },
        audio: false
      });
      const video = document.getElementById('nav-video');
      video.srcObject = mediaStream;
      await video.play();
      active = true;
      document.getElementById('nav-badge').textContent = '● LIVE';
      document.getElementById('nav-badge').classList.add('live');
      document.getElementById('nav-start-btn').disabled = true;
      document.getElementById('nav-stop-btn').disabled = false;
      captureLoop();
    } catch(e) {
      document.getElementById('nav-det-text').textContent = 'Webcam error: ' + e.message;
    }
  };

  window.navStopStream = function() {
    active = false;
    if (mediaStream) mediaStream.getTracks().forEach(t => t.stop());
    document.getElementById('nav-badge').textContent = '● IDLE';
    document.getElementById('nav-badge').classList.remove('live');
    document.getElementById('nav-start-btn').disabled = false;
    document.getElementById('nav-stop-btn').disabled = true;
    document.getElementById('nav-det-text').textContent = 'detections: --';
  };

  // ── Called by Python kernel with annotated frame ───────
  window.navUpdate = function(b64, count, detStr, frameNum) {
    document.getElementById('nav-img').src = 'data:image/jpeg;base64,' + b64;
    document.getElementById('nav-det-text').textContent =
      count > 0 ? 'detected: ' + detStr : 'detections: none';
    document.getElementById('nav-frame-num').textContent = 'frame: ' + frameNum;
  };

  window.navError = function(msg) {
    document.getElementById('nav-det-text').textContent = 'error: ' + msg;
  };

})();
</script>
'''))

print('UI ready ✓')
print('Click ▶ START STREAM in the widget above and allow webcam access.')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 7 - Static Image Test (optional)              ║
# ╚══════════════════════════════════════════════════════╝
# Test YOLO on a local image before going live.
# Set TEST_IMAGE to a path, or leave None to skip.

TEST_IMAGE = None  # e.g. '/root/test.jpg'

if TEST_IMAGE:
    frame   = cv2.imread(TEST_IMAGE)
    results = model(frame, verbose=True, device=device)
    annotated, det_count, dets = draw_detections(frame, results)

    rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    buf = BytesIO()
    Image.fromarray(rgb).save(buf, format='JPEG', quality=92)
    b64 = base64.b64encode(buf.getvalue()).decode()

    display(HTML(
        f'<img src="data:image/jpeg;base64,{b64}"'
        f' style="max-width:800px;border-radius:8px;">'
    ))
    print(f'Detected {det_count} object(s):')
    for d in dets:
        print(f"  {d['class']:15s} {d['conf']:.1%}")
else:
    print('No test image set. Set TEST_IMAGE = "/path/to/image.jpg" to use this cell.')